In [ ]:
import pandas as pd
import numpy as np
import random
from copy import deepcopy

In [ ]:
def build_valid_fpl_squad(df, gw, min_minutes=30):
    gw_df = df[(df["round"] == gw) & (df["minutes"] >= min_minutes)]

    def select_players(pos, count):
        pool = gw_df[gw_df["position"] == pos]
        return list(pool.sample(n=count, replace=False)["name"])

    squad = {
        "GK": select_players("GK", 2),
        "DEF": select_players("DEF", 5),
        "MID": select_players("MID", 5),
        "FWD": select_players("FWD", 3)
    }

    return squad


In [ ]:
def flatten_squad(squad):
    return [player for pos in squad for player in squad[pos]]

In [ ]:
def generate_trade_options(df, current_player, gw, top_k=3):
    # Get position and predicted points
    player_rows = df[(df["name"] == current_player) & (df["round"] == gw)]
    if player_rows.empty:
        return []  # No trade suggestions if player has no data for this round

    pos = player_rows["position"].values[0]
    curr_pred = player_rows["total_points"].values[0]

    # Filter pool for same-position players in the same round
    pool = df[(df["position"] == pos) & (df["round"] == gw)]
    pool = pool[pool["name"] != current_player].copy()

    # Score similarity by absolute difference
    pool["similarity"] = (pool["total_points"] - curr_pred).abs()
    pool = pool.sort_values(by="similarity")

    return pool["name"].head(top_k).tolist()

In [ ]:
class SoloFPLTransferEnv:
    def __init__(self, df, gw_start=1):
        self.df = df
        self.current_gw = gw_start
        self.squad = build_valid_fpl_squad(df, self.current_gw)
        self.state = flatten_squad(self.squad)
        self.history = []

    def reset(self):
        self.current_gw = 1
        self.squad = build_valid_fpl_squad(self.df, self.current_gw)
        self.state = flatten_squad(self.squad)
        self.history = []
        return self.state

    def step(self, action):
        """
        action: tuple (player_out, player_in)
        """
        player_out, player_in = action
        squad_flat = flatten_squad(self.squad)

        # Simulate points with current squad
        curr_df = self.df[self.df["round"] == self.current_gw]
        original_points = curr_df[curr_df["name"].isin(squad_flat)]["total_points"].sum()

        # Make the trade
        for pos in self.squad:
            if player_out in self.squad[pos]:
                self.squad[pos].remove(player_out)
                self.squad[pos].append(player_in)
                break

        new_flat = flatten_squad(self.squad)
        new_points = curr_df[curr_df["name"].isin(new_flat)]["total_points"].sum()
        reward = new_points - original_points

        self.history.append((self.current_gw, player_out, player_in, reward))
        self.current_gw += 1
        done = self.current_gw > self.df["round"].max()
        self.state = new_flat

        return self.state, reward, done, {}

    def valid_trade_actions(self):
        """
        Generate all trade actions (player_out, player_in)
        """
        trades = []
        for player in flatten_squad(self.squad):
            options = generate_trade_options(self.df, player, self.current_gw)
            for opt in options:
                trades.append((player, opt))
        return trades